# OPSD gap on GSM8K

Tufa Labs showed that the **student vs self-teacher accuracy gap**, measured before
any training, linearly predicts how much OPSD post-training will help
([arXiv:2605.30070](https://arxiv.org/abs/2605.30070)). They measured it on code.

This notebook measures the same predictor on **grade-school math**, and adds two
contexts their setup does not have: a **misleading** one and a **corrupted** one.

No training here. What is measured is the predictor, not the final improvement.

**Runtime -> Change runtime type -> GPU (T4).**

## 1. Setup

In [ ]:
!git clone -q https://github.com/kaj04/opsd-gap-gsm8k.git /content/opsd-gap-gsm8k || echo 'already cloned'
%cd /content/opsd-gap-gsm8k
!pip install -q vllm datasets
# pip warns about conflicts with Colab's preinstalled RAPIDS packages: harmless.
# Then Runtime -> Restart session, and run the next cell.

In [ ]:
%cd /content/opsd-gap-gsm8k
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. The task

GSM8K: grade-school word problems with a human-written step-by-step solution and a
single numeric answer. The solutions are what every privileged context below is
built from.

In [ ]:
from datasets import load_dataset
import numpy as np, textwrap

ds = load_dataset("openai/gsm8k", "main", split="test")
r = ds[3]
print(textwrap.fill(r["question"], 78))
print("\nreference solution:")
print(r["answer"])

sub = ds.select(range(300))
steps = [x["answer"].count("<<") for x in sub]
print("\nover the 300 problems used here: median %d calculation steps (min %d, max %d)" % (
    np.median(steps), min(steps), max(steps)))

## 3. The six conditions

The self-teacher is the same model, with extra text prepended. The teacher template
is copied verbatim from Tufa's `actor.yaml`; the contexts are ours.

| condition | what the teacher gets | expectation |
|---|---|---|
| `none` | nothing | the ruler: plain resampling |
| `gold_step1` | the first reasoning step | small help |
| `gold_steps` | every step but the last, so no final number | more help |
| `gt_solution` | the full solution, answer included | ceiling |
| `wrong_solution` | another problem's solution | should hurt |
| `corrupted_solution` | this problem's solution with the numbers shifted | should hurt, and is harder to notice |

The cell below prints one real example of each.

In [ ]:
!python src/inspect_prompts.py --index 3

## 4. Run

300 problems, 8 rollouts each, `enable_thinking=False`, temperature 0.7.

Following Tufa's gating, privileged context goes only to rollouts the student got
wrong, so the teacher is only run on those. Takes roughly 45 minutes on a T4.

In [ ]:
!python src/run_eval.py --n-problems 300 --n-samples 8 --tag run

## 5. Results

In [ ]:
!python src/analyze.py --tag run
from IPython.display import Image
Image('figures/gaps.png')

In [ ]:
# summary table
import json
import pandas as pd

WHAT = {
    "none": "nothing (plain resampling)",
    "gold_step1": "first reasoning step",
    "gold_steps": "all steps but the last",
    "gt_solution": "full solution, answer included",
    "wrong_solution": "another problem's solution",
    "corrupted_solution": "same solution, numbers shifted",
}

s = json.load(open("results/run_summary.json"))
df = pd.read_csv("results/gaps.csv")
df["what the teacher sees"] = df["condition"].map(WHAT)
df["95% CI"] = df.apply(lambda r: "[%+.3f, %+.3f]" % (r.ci_lo, r.ci_hi), axis=1)
out = df[["condition", "what the teacher sees", "recovery",
          "diff_vs_none", "95% CI", "follow_context"]]
out = out.rename(columns={"diff_vs_none": "vs none",
                          "follow_context": "copies the context"})
out = out.sort_values("vs none", ascending=False).reset_index(drop=True)

print("%s, %d problems x %d rollouts, student accuracy %.3f, %d failed rollouts\n" % (
    s["model"], s["n_problems"], s["n_samples"], s["student_acc"], s["n_failed"]))
display(out.round(3))
print("\nrecovery           share of the student's failed rollouts the self-teacher fixes")
print("vs none            the same number minus plain resampling, with a bootstrap CI")
print("copies the context how often the model returned the number the context points at")

## 6. Save the results back to the repo

Download, unzip locally, commit. Then `File -> Save a copy in GitHub` to push this
notebook with its outputs.

In [ ]:
from google.colab import files
!zip -qr results.zip results figures data/contexts
files.download('results.zip')